# Model Training, Tuning, and Evaluation
This notebook implements feature engineering, train-test splitting, hyperparameter tuning, model training (traditional models, ML regressors, deep learning), and automated best-model selection.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.data_preprocessing import load_data, clean_data, detect_dataset_type, preprocess_and_scale
from src.feature_engineering import engineer_features
from src.train_model import train_regression_model, train_traditional_time_series, train_lstm_gru, save_pipeline
from src.evaluate import evaluate_model, compile_comparison, analyze_residuals

### 1. Load, Preprocess, and Engineer Features

In [ ]:
raw_path = '../data/raw/default_stock_data.csv'
df_raw = load_data(raw_path)
dataset_type, datetime_col = detect_dataset_type(df_raw)
df_clean = clean_data(df_raw, datetime_col=datetime_col)

# Identify target column
target_col = 'Close' if 'Close' in df_clean.columns else df_clean.columns[-1]
is_ts = (dataset_type == 'time_series')

# Generate features
df_feat = engineer_features(df_clean, target_col=target_col, is_time_series=is_ts)
print(f"Feature matrix shape: {df_feat.shape}")
df_feat.head()

### 2. Split Dataset and Standardize Features

In [ ]:
# For time series, split chronologically; for regression, split randomly
if is_ts:
    train_size = int(len(df_feat) * 0.8)
    df_train = df_feat.iloc[:train_size]
    df_test = df_feat.iloc[train_size:]
else:
    from sklearn.model_selection import train_test_split
    df_train, df_test = train_test_split(df_feat, test_size=0.2, random_state=42)

# Preprocess and scale features
X_train, y_train, encoders, scaler = preprocess_and_scale(df_train, target_col=target_col, scaling_method='standard')
X_test, y_test, _, _ = preprocess_and_scale(df_test, target_col=target_col, encoders=encoders, scaler=scaler)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

### 3. Model Training & Evaluation
We will train Random Forest and XGBoost Regressors (common to both dataset types) and evaluate their performance.

In [ ]:
models_to_test = ['RandomForest', 'XGBoost']
results = {}
trained_models = {}

for m_name in models_to_test:
    print(f"Training {m_name}...")
    # Train model (with tuning)
    model = train_regression_model(X_train, y_train, model_name=m_name, tune=True)
    trained_models[m_name] = model
    
    # Predict and evaluate
    preds = model.predict(X_test)
    metrics = evaluate_model(y_test, preds, is_time_series=is_ts)
    results[m_name] = metrics

# Display comparison
df_compare = compile_comparison(results, is_time_series=is_ts)
df_compare

### 4. Residual Analysis of Best Model

In [ ]:
best_model_name = df_compare.index[0]
best_model = trained_models[best_model_name]
test_preds = best_model.predict(X_test)

residuals, stats = analyze_residuals(y_test, test_preds)
print(f"Best Model: {best_model_name}")
print(f"Residual Stats: {stats}")

plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=30, color='salmon', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.title('Residuals Histogram (Distribution of Errors)')
plt.xlabel('Error')
plt.ylabel('Frequency')
plt.show()

### 5. Save the Winning Model
We package the best model, the scaler, the encoders, and the features list into a pipeline dictionary, then save it.

In [ ]:
pipeline_dict = {
    'model_type': dataset_type,
    'model_name': best_model_name,
    'model': best_model,
    'scaler': scaler,
    'encoders': encoders,
    'target_col': target_col,
    'datetime_col': datetime_col,
    'features': list(X_train.columns),
    'metrics': results[best_model_name]
}

save_path = '../models/saved_model.pkl'
save_pipeline(pipeline_dict, save_path)